In [1]:
import pandas as pd
import numpy as np
import itertools
import pymc as pm
import arviz as az

import sys
sys.path.append('../Source')
from Pipeline.build_country_comparisons import buildCountryComparisons
from Pipeline.uncertainty_metrics import computeUncertainty

alignedCountryData = pd.read_csv('../Data/Processed/alignedDataCountry.csv')
countryComparisons = buildCountryComparisons(alignedCountryData)

print(countryComparisons.shape)

(232525, 6)


In [2]:
caucasusData = countryComparisons[countryComparisons['region'] == 'Caucasus and Central Asia']

caucasusWeeks = sorted(caucasusData['week'].unique())
print(len(caucasusWeeks))
print(caucasusWeeks[:5])

391
[Period('2018-01-01/2018-01-07', 'W-SUN'), Period('2018-01-08/2018-01-14', 'W-SUN'), Period('2018-01-15/2018-01-21', 'W-SUN'), Period('2018-01-22/2018-01-28', 'W-SUN'), Period('2018-01-29/2018-02-04', 'W-SUN')]


In [4]:
regionCountries = sorted(set(caucasusData['countryI']) | set(caucasusData['countryJ']))
regionIndexMap = {country: i for i, country in enumerate(regionCountries)}

nRegionCountries = len(regionCountries)
print(nRegionCountries)
print(regionCountries)

9
['Afghanistan', 'Armenia', 'Azerbaijan', 'Georgia', 'Kazakhstan', 'Kyrgyzstan', 'Tajikistan', 'Turkmenistan', 'Uzbekistan']


In [5]:
initialWeekData = caucasusData[caucasusData['week'] == caucasusWeeks[0]]

initialIndexI = initialWeekData['countryI'].map(regionIndexMap).to_numpy()
initialIndexJ = initialWeekData['countryJ'].map(regionIndexMap).to_numpy()
initialWinsI = initialWeekData['winsI'].to_numpy()
initialTotals  = initialWeekData['total'].to_numpy()

with pm.Model() as rollingModel:
    indexIData = pm.Data('indexIData', initialIndexI)
    indexJData = pm.Data('indexJData', initialIndexJ)
    winsIData = pm.Data('winsIData', initialWinsI)
    totalsData = pm.Data('totalsData', initialTotals)

    beta = pm.ZeroSumNormal('beta', sigma=1.0, shape=nRegionCountries)
    p = pm.math.sigmoid(beta[indexIData] - beta[indexJData])
    obs = pm.Binomial('obs', n=totalsData, p=p, observed=winsIData)

In [8]:
import time

weeklyResults = []
weeksSkipped = []

startTime = time.time()

for week in caucasusWeeks:
    weekData = caucasusData[caucasusData['week'] == week]

    if len(weekData) == 0:
        weeksSkipped.append(week)
        continue

    indexI = weekData['countryI'].map(regionIndexMap).to_numpy()
    indexJ = weekData['countryJ'].map(regionIndexMap).to_numpy()
    winsI = weekData['winsI'].to_numpy()
    totals = weekData['total'].to_numpy()

    with rollingModel:
        pm.set_data({
            'indexIData': indexI,
            'indexJData': indexJ,
            'winsIData': winsI,
            'totalsData': totals
        })

        trace = pm.sample(200, tune=200, target_accept=0.9, return_inferencedata=True, chains=2, cores=2, progressbar=False)

    weeklyResults.append({'week': week, 'trace': trace})

endTime = time.time()

print(f"Total time: {endTime - startTime:.1f} seconds")
print(f"Weeks fit: {len(weeklyResults)}")
print(f"Weeks skipped: {len(weeksSkipped)}")

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [beta]
Sampling 2 chains for 200 tune and 200 draw iterations (400 + 400 draws total) took 14 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [beta]
Sampling 2 chains for 200 tune and 200 draw iterations (400 + 400 draws total) took 15 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher

KeyboardInterrupt: 